# Chapter 22 Walkthrough: Linear & Regularized Regression (OLS, Ridge, Lasso, ElasticNet)

**Book Reference:** Chapter 22, Sections 22.1 – 22.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every analysis uses the bundled `mch_survey_sample` maternal and child health dataset.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & MCH Survey Data
!pip install -q sharukh-tutor
import numpy as np
import pandas as pd
from sklearn.linear_model import (
    ElasticNet, Lasso, LinearRegression, Ridge
)
from sklearn.metrics import (
    mean_absolute_error, r2_score, root_mean_squared_error
)
from sklearn.model_selection import (
    KFold, cross_val_score, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = mch["residence"].astype(str).str.strip().str.title()
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)

feat_cols = [
    "anc_visits", "maternal_hb_g_dl", "maternal_age_years", "is_urban"
]
clean = mch[feat_cols + ["child_haz"]].dropna()
print(f"Raw rows: {len(mch_raw)} -> Complete rows: {len(clean)}")
print(clean.mean().round(3).to_string())


## Section 22.3.1 — Regression Metrics: MSE, RMSE, MAE & $R^2$

In [ ]:
X = clean[feat_cols]
y = clean["child_haz"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

ols_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ols", LinearRegression()),
])
ols_pipe.fit(X_tr, y_tr)
y_pred = ols_pipe.predict(X_te)

err = y_te.to_numpy() - y_pred
mse = float(np.mean(err ** 2))
rmse = float(np.sqrt(mse))
mae = float(np.mean(np.abs(err)))
ss_tot = float(np.sum((y_te.to_numpy() - y_te.mean()) ** 2))
r2 = 1.0 - float(np.sum(err ** 2)) / ss_tot

print(f"From scratch -> MSE={mse:.4f}, RMSE={rmse:.4f}, "
      f"MAE={mae:.4f}, R2={r2:.4f}")
print(f"Scikit-Learn -> RMSE={root_mean_squared_error(y_te, y_pred):.4f}, "
      f"MAE={mean_absolute_error(y_te, y_pred):.4f}, "
      f"R2={r2_score(y_te, y_pred):.4f}")


## Section 22.3.2 — Closed-Form OLS: The Normal Equation From Scratch

In [ ]:
# Standardize using training-fold statistics ONLY (zero leakage)
mu_tr = X_tr.to_numpy().mean(axis=0)
sd_tr = X_tr.to_numpy().std(axis=0)
Z_tr = (X_tr.to_numpy() - mu_tr) / sd_tr
y_tr_arr = y_tr.to_numpy()

Zb = np.column_stack([np.ones(len(Z_tr)), Z_tr])
theta_lstsq = np.linalg.lstsq(Zb, y_tr_arr, rcond=None)[0]
theta_pinv = np.linalg.pinv(Zb.T @ Zb) @ Zb.T @ y_tr_arr

sk_ols = ols_pipe.named_steps["ols"]
print(f"Normal Eq (lstsq) b={theta_lstsq[0]:.4f}, "
      f"w={np.round(theta_lstsq[1:], 4)}")
print(f"Normal Eq (pinv)  b={theta_pinv[0]:.4f}, "
      f"w={np.round(theta_pinv[1:], 4)}")
print(f"Scikit-Learn OLS  b={sk_ols.intercept_:.4f}, "
      f"w={np.round(sk_ols.coef_, 4)}")


## Section 22.3.3 — Pure-NumPy Batch Gradient Descent on MSE

In [ ]:
def run_batch_gd(Z, y_vec, lr=0.05, epochs=300):
    n, p = Z.shape
    w = np.zeros(p, dtype=float)
    b = 0.0
    history = []
    for _ in range(epochs):
        err = (Z @ w + b) - y_vec
        history.append(float(np.mean(err ** 2)))
        w = w - lr * ((2.0 / n) * (Z.T @ err))
        b = b - lr * ((2.0 / n) * float(np.sum(err)))
    return w, b, history

w_gd, b_gd, losses = run_batch_gd(Z_tr, y_tr_arr, lr=0.05, epochs=300)
print(f"MSE Epoch 1: {losses[0]:.4f} -> Epoch 25: {losses[24]:.4f} "
      f"-> Epoch 300: {losses[-1]:.4f}")
print(f"Batch GD params   b={b_gd:.4f}, w={np.round(w_gd, 4)}")
max_diff = float(np.max(np.abs(w_gd - sk_ols.coef_)))
print(f"Max |w_gd - w_ols|: {max_diff:.6f}")


## Section 22.3.4 — Multicollinearity Instability: OLS Explosion vs. Ridge Stabilization

In [ ]:
rng = np.random.default_rng(42)
X_col = X_tr.copy()
# Add a nearly collinear duplicate clinic-visit counter (r > 0.9999)
X_col["anc_proxy"] = (
    X_col["anc_visits"] + rng.normal(0.0, 0.01, size=len(X_col))
)

for label, est in [
    ("OLS (unregularized)", LinearRegression()),
    ("Ridge (alpha=1.0)  ", Ridge(alpha=1.0)),
    ("Ridge (alpha=10.0) ", Ridge(alpha=10.0)),
]:
    pipe = Pipeline([("scaler", StandardScaler()), ("m", est)])
    pipe.fit(X_col, y_tr)
    c = pipe.named_steps["m"].coef_
    l2_norm = float(np.linalg.norm(c))
    print(f"{label} anc={c[0]:+7.3f}, proxy={c[4]:+7.3f} "
          f"| ||w||_2={l2_norm:.3f}")


## Section 22.3.5 — Regularization Geometry: Ridge ($L_2$) vs. Lasso ($L_1$) vs. ElasticNet

In [ ]:
models = [
    ("OLS", LinearRegression()),
    ("Ridge(1.0)", Ridge(alpha=1.0)),
    ("Ridge(10.0)", Ridge(alpha=10.0)),
    ("Lasso(0.05)", Lasso(alpha=0.05, max_iter=5000)),
    ("ElasticNet(0.05)", ElasticNet(alpha=0.05, l1_ratio=0.5,
                                    max_iter=5000)),
]
rows = []
for name, est in models:
    pipe = Pipeline([("scaler", StandardScaler()), ("m", est)])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    c = pipe.named_steps["m"].coef_
    rows.append({
        "model": name,
        "rmse": round(float(root_mean_squared_error(y_te, pred)), 4),
        "nonzero": int(np.sum(np.abs(c) > 1e-6)),
        "anc": round(float(c[0]), 4),
        "hb": round(float(c[1]), 4),
        "age": round(float(c[2]), 4),
        "urban": round(float(c[3]), 4),
    })
print(pd.DataFrame(rows).to_string(index=False))


## Section 22.3.6 — Residual Diagnostics & Error Inspection

In [ ]:
lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(alpha=0.05, max_iter=5000)),
])
lasso_pipe.fit(X_tr, y_tr)
y_hat_lasso = lasso_pipe.predict(X_te)
resid = y_te.to_numpy() - y_hat_lasso

within_2sd = float(np.mean(np.abs(resid) <= 2.0 * np.std(resid)))
print(f"Mean residual : {np.mean(resid):.4f}")
print(f"Std residual  : {np.std(resid):.4f}")
print(f"Max |error|   : {np.max(np.abs(resid)):.4f}")
print(f"Share <= 2 SD : {within_2sd * 100:.1f}%")


## Section 22.3.7 — Break It On Purpose! (Regression Error Anatomy)

In [ ]:
# Intentional Bug #1: LinAlgError — inverting singular collinear X.T @ X
X_sing = np.array([[1.0, 2.0, 4.0], [1.0, 3.0, 6.0], [1.0, 5.0, 10.0]])
np.linalg.inv(X_sing.T @ X_sing)


In [ ]:
# Intentional Bug #2: ValueError — passing 1D X instead of 2D (n, p)
x_1d = clean["anc_visits"].to_numpy()[:5]
y_1d = clean["child_haz"].to_numpy()[:5]
LinearRegression().fit(x_1d, y_1d)


In [ ]:
# Intentional Bug #3: ValueError — negative regularization penalty alpha
Ridge(alpha=-1.0).fit(X_tr, y_tr)


## Section 22.4 — Stage 4 Blueprint: Leak-Free CV Tuning & Regression Benchmark Pipeline

In [ ]:
def build_haz_regression_report(raw_df, alphas=(0.1, 1.0, 10.0, 100.0)):
    """Clean MCH data, tune Ridge via 5-fold CV, and benchmark models."""
    df = raw_df.copy()
    df["residence"] = df["residence"].astype(str).str.strip().str.title()
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    feats = [
        "anc_visits", "maternal_hb_g_dl",
        "maternal_age_years", "is_urban",
    ]
    cdf = df[feats + ["child_haz"]].dropna()
    X_all, y_all = cdf[feats], cdf["child_haz"]
    X_a, X_b, y_a, y_b = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42
    )

    # 1. Leak-free 5-fold CV on training split ONLY
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_rows = []
    for a in alphas:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=float(a))),
        ])
        rmses = -cross_val_score(
            pipe, X_a, y_a, cv=kf,
            scoring="neg_root_mean_squared_error",
        )
        cv_rows.append({
            "alpha": float(a),
            "cv_rmse": round(float(rmses.mean()), 4),
        })
    cv_df = pd.DataFrame(cv_rows)
    best_a = float(cv_df.loc[cv_df["cv_rmse"].idxmin(), "alpha"])

    # 2. Final holdout evaluation with tuned Ridge
    best_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=best_a)),
    ])
    best_pipe.fit(X_a, y_a)
    preds = best_pipe.predict(X_b)
    test_rmse = round(float(root_mean_squared_error(y_b, preds)), 4)
    resid = y_b.to_numpy() - preds
    diag = {
        "mean_resid": round(float(np.mean(resid)), 4),
        "max_abs_err": round(float(np.max(np.abs(resid))), 4),
    }
    return len(cdf), cv_df, best_a, test_rmse, diag

n_ok, cv_tbl, best_alpha, holdout_rmse, resid_diag = (
    build_haz_regression_report(mch_raw)
)
print(f"Clean rows: {n_ok} | Best CV alpha: {best_alpha} "
      f"| Test RMSE: {holdout_rmse}")
print(cv_tbl.to_string(index=False))
print("Residual diagnostics:", resid_diag)
